# Pre procesamiento

## Imports de las librerías

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

import pandas as pd
from IPython.display import display

from src.config import ProjectConfig
from src.data_prep import WineDataLoader, WinePreprocessor

ProjectConfig.create_directories()

## Cargar y unificar datos

In [3]:
loader = WineDataLoader(
    red_wine_path=ProjectConfig.RED_WINE_FILE,
    white_wine_path=ProjectConfig.WHITE_WINE_FILE,
)

wine_data = loader.combine_data()

print("Dimensión original:", wine_data.shape)
display(wine_data.head())

Dimensión original: (6497, 13)


,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,wine_type
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,red
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5,red
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5,red
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6,red
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,red


## Crear el preprocesador

In [4]:
preprocessor = WinePreprocessor(
    dataframe=wine_data,
    high_quality_threshold=(
        ProjectConfig.HIGH_QUALITY_THRESHOLD
    ),
    test_size=ProjectConfig.TEST_SIZE,
    random_state=ProjectConfig.RANDOM_STATE,
)

## Limpiar datos

In [5]:
rows_before = len(preprocessor.dataframe)

clean_data = preprocessor.clean_data()

rows_after = len(clean_data)

print("Registros antes de la limpieza:", rows_before)
print("Registros después de la limpieza:", rows_after)
print("Registros removidos:", rows_before - rows_after)
print("Valores faltantes:", clean_data.isna().sum().sum())

Registros antes de la limpieza: 6497
Registros después de la limpieza: 5320
Registros removidos: 1177
Valores faltantes: 0


## Crear high_quality ) > 7

In [6]:
target_data = preprocessor.create_target_variables()

quality_distribution = (
    target_data["high_quality"]
    .value_counts()
    .rename(index={0: "No alta", 1: "Alta"})
    .to_frame("count")
)

quality_distribution["percentage"] = (
    quality_distribution["count"]
    / quality_distribution["count"].sum()
) * 100

display(quality_distribution)

,count,percentage
high_quality,,
No alta,4311,81.033835
Alta,1009,18.966165


## Codificar el tipo de Vino (en términos de wine_type_white)

In [ ]:
encoded_data = preprocessor.encode_wine_type()

In [11]:

display(encoded_data.head())
print(encoded_data.dtypes)

,fixed_acidity,volatile_acidity,citric_acid,residual_sugar,chlorides,free_sulfur_dioxide,total_sulfur_dioxide,density,ph,sulphates,alcohol,quality,high_quality,wine_type_white
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,0,0
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5,0,0
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5,0,0
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6,0,0
4,7.4,0.66,0.00,1.8,0.075,13.0,40.0,0.9978,3.51,0.56,9.4,5,0,0


fixed_acidity           float64
volatile_acidity        float64
citric_acid             float64
residual_sugar          float64
chlorides               float64
free_sulfur_dioxide     float64
total_sulfur_dioxide    float64
density                 float64
ph                      float64
sulphates               float64
alcohol                 float64
quality                   int64
high_quality              int64
wine_type_white           int64
dtype: object


## Dividir los datos ( Train y Test )


In [13]:
split_data = preprocessor.split_data()

print("X_train:", split_data["X_train"].shape)
print("X_test:", split_data["X_test"].shape)

print(
    "y_train_regression:",
    split_data["y_train_regression"].shape,
)

print(
    "y_train_classification:",
    split_data["y_train_classification"].shape,
)

X_train: (4256, 12)
X_test: (1064, 12)
y_train_regression: (4256,)
y_train_classification: (4256,)


Verificar la estratificación

In [14]:
print("Distribución general:")
display(
    encoded_data["high_quality"]
    .value_counts(normalize=True)
    .sort_index()
)

print("Distribución en entrenamiento:")
display(
    split_data["y_train_classification"]
    .value_counts(normalize=True)
    .sort_index()
)

print("Distribución en prueba:")
display(
    split_data["y_test_classification"]
    .value_counts(normalize=True)
    .sort_index()
)

Distribución general:


high_quality
0    0.810338
1    0.189662
Name: proportion, dtype: float64

Distribución en entrenamiento:


high_quality
0    0.810385
1    0.189615
Name: proportion, dtype: float64

Distribución en prueba:


high_quality
0    0.81015
1    0.18985
Name: proportion, dtype: float64

## Normalizar

In [15]:
normalized_data = preprocessor.normalize_data(
    split_data
)

display(normalized_data["X_train"].head())

,fixed_acidity,volatile_acidity,citric_acid,residual_sugar,chlorides,free_sulfur_dioxide,total_sulfur_dioxide,density,ph,sulphates,alcohol,wine_type_white
4697,-0.994120,-0.505515,-0.542621,-0.592638,-0.288436,-0.164518,0.319999,-0.743530,2.569540,1.328492,0.635962,0.584586
2453,0.061465,0.451759,0.406770,0.382845,-0.314551,-0.053183,0.144239,-0.689620,-0.214790,-1.092401,1.732778,0.584586
1298,-0.466327,0.571418,-1.559825,-0.636978,0.155518,-1.166538,-1.683656,-0.035963,1.240655,1.059504,0.635962,-1.710614
3478,-0.692524,-1.283300,0.678025,-0.880849,-0.366781,1.839521,0.566061,-1.309584,-0.594471,1.261245,0.551592,0.584586
1568,-1.220317,-0.924322,-0.406993,-0.814338,-0.706275,-0.999534,-0.664253,-1.363493,0.164891,-1.092401,0.298480,0.584586


## Validar normalizacion

In [17]:
normalization_check = pd.DataFrame(
    {
        "mean": normalized_data["X_train"].mean(),
        "standard_deviation":
            normalized_data["X_train"].std(ddof=0),
    }
)

display(normalization_check)

,mean,standard_deviation
fixed_acidity,3.823174e-16,1.0
volatile_acidity,3.339017e-17,1.0
citric_acid,1.419082e-17,1.0
residual_sugar,8.013640e-17,1.0
chlorides,1.335607e-16,1.0
free_sulfur_dioxide,3.005115e-17,1.0
total_sulfur_dioxide,9.683148e-17,1.0
density,2.228460e-14,1.0
ph,4.591148e-16,1.0
sulphates,-3.188761e-16,1.0


## Guardar resultados

In [18]:
encoded_data.to_csv(
    ProjectConfig.CLEAN_DATA_FILE,
    index=False,
)

preprocessor.save_processed_data(
    split_data=normalized_data,
    output_directory=ProjectConfig.PROCESSED_DATA_DIR,
    scaler_path=ProjectConfig.SCALER_FILE,
)

print("Datos procesados guardados correctamente.")
print("Escalador guardado en:", ProjectConfig.SCALER_FILE)

Datos procesados guardados correctamente.
Escalador guardado en: C:\Proyecto_D_Calidad_Vino\models\standard_scaler.joblib
